#### Given a dataset with raw attributes (e.g., sales transactions), create new features (e.g., total revenue, customer frequency, category-based sales). Apply discretization, transformation, and outlier detection to improve dataset quality.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings

# Suppress warnings for cleaner output (e.g., for pd.to_datetime('now'))
warnings.filterwarnings('ignore')

In [ ]:
#%%
# Create a raw dataset of sales transactions
data = {
    'invoice_id': ['A1001', 'A1001', 'A1002', 'A1003', 'A1003', 'A1003', 
                   'A1004', 'A1005', 'A1005', 'A1006', 'A1007', 'A1008'],
    'customer_id': [101, 101, 102, 101, 101, 101, 
                    103, 102, 102, 101, 104, 104],
    'product_category': ['Electronics', 'Clothing', 'Electronics', 'Books', 'Clothing', 'Home', 
                         'Beauty', 'Electronics', 'Home', 'Books', 'Electronics', 'Electronics'],
    'quantity': [1, 2, 1, 3, 1, 1, 
                 2, 1, 5, 1, 1, 1],
    'unit_price': [120, 30, 150, 15, 25, 50, 
                   20, 130, 40, 18, 5000, 4800], # Added large values for outlier detection
    'invoice_date': [
        '2023-01-10', '2023-01-10', '2023-01-11', '2023-01-15', '2023-01-15', '2023-01-15',
        '2023-01-20', '2023-01-22', '2023-01-22', '2023-02-01', '2023-02-05', '2023-02-06'
    ]
}

df = pd.DataFrame(data)
df['invoice_date'] = pd.to_datetime(df['invoice_date'])

print("--- Raw Transactional Data ---")
print(df)
df.info()

In [ ]:
#%%
# Create a total_price column (quantity * unit_price)
df['total_price'] = df['quantity'] * df['unit_price']

print("--- Data with total_price ---")
print(df[['invoice_id', 'product_category', 'quantity', 'unit_price', 'total_price']])

In [ ]:
#%%
# Define a "snapshot date" to calculate recency (e.g., the day after the last transaction)
snapshot_date = df['invoice_date'].max() + pd.Timedelta(days=1)
print(f"Snapshot date for recency: {snapshot_date.date()}")

# Aggregate data at the customer level
customer_df = df.groupby('customer_id').agg(
    total_revenue=('total_price', 'sum'),
    customer_frequency=('invoice_id', 'nunique'),
    most_recent_purchase=('invoice_date', 'max')
)

# Calculate recency (days_since_last_purchase)
customer_df['days_since_last_purchase'] = (snapshot_date - customer_df['most_recent_purchase']).dt.days

# Drop the original date column as it's no longer needed
customer_df = customer_df.drop(columns=['most_recent_purchase'])

print("\n--- Aggregated Customer Data ---")
print(customer_df)

In [ ]:
#%%
# Create category-based sales features using pivot_table
category_sales = df.pivot_table(
    index='customer_id',
    columns='product_category',
    values='total_price',
    aggfunc='sum'
).fillna(0) # Fill NaNs with 0 for customers who didn't buy in a category

# Add a prefix to the column names for clarity
category_sales = category_sales.add_prefix('sales_')

print("--- Customer Sales by Category ---")
print(category_sales)

# Merge this back into our main customer_df
customer_df = customer_df.join(category_sales)

print("\n--- Combined Customer Data ---")
print(customer_df)

In [ ]:
#%%
# Discretize 'customer_frequency' into groups
# We'll define bins: 1 = Low, 2 = Medium, 3+ = High
bins = [0, 1, 2, np.inf]
labels = ['Low', 'Medium', 'High']

customer_df['frequency_group'] = pd.cut(
    customer_df['customer_frequency'], 
    bins=bins, 
    labels=labels, 
    right=True
)

print("--- Data with Discretized Frequency ---")
print(customer_df[['customer_frequency', 'frequency_group']])

In [ ]:
#%%
# 1. Log Transformation for skewed data (e.g., revenue)
# We use np.log1p which calculates log(1 + x) to handle potential zero values
customer_df['log_revenue'] = np.log1p(customer_df['total_revenue'])

print("\n--- Log-Transformed Revenue ---")
print(customer_df[['total_revenue', 'log_revenue']])

# 2. Standardization (Z-score scaling)
scaler = StandardScaler()

# We fit_transform the scaler. Note the .values and reshaping.
scaled_revenue = scaler.fit_transform(customer_df[['total_revenue']].values)
customer_df['scaled_revenue'] = scaled_revenue

print("\n--- Standard-Scaled Revenue ---")
print(customer_df[['total_revenue', 'scaled_revenue']])

In [ ]:
#%%
# Outlier Detection for 'total_revenue' using the IQR method
feature = 'total_revenue'
Q1 = customer_df[feature].quantile(0.25)
Q3 = customer_df[feature].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"--- Outlier Detection for '{feature}' ---")
print(f"Q1: {Q1:.2f}")
print(f"Q3: {Q3:.2f}")
print(f"IQR: {IQR:.2f}")
print(f"Lower Bound (Outliers <): {lower_bound:.2f}")
print(f"Upper Bound (Outliers >): {upper_bound:.2f}")

# Identify and flag the outliers
customer_df['is_revenue_outlier'] = (customer_df[feature] < lower_bound) | (customer_df[feature] > upper_bound)

print("\n--- Customers Flagged as Outliers ---")
print(customer_df[customer_df['is_revenue_outlier']])

In [ ]:
#%%
print("\n--- FINAL PROCESSED CUSTOMER DATASET ---")

# Re-ordering columns for a cleaner view
all_columns = [
    # Original Aggregated
    'total_revenue', 
    'customer_frequency', 
    'days_since_last_purchase',
    # Category Sales
    'sales_Beauty', 
    'sales_Books', 
    'sales_Clothing', 
    'sales_Electronics', 
    'sales_Home',
    # Discretized
    'frequency_group',
    # Transformed
    'log_revenue', 
    'scaled_revenue',
    # Outlier Flag
    'is_revenue_outlier'
]

# Ensure all columns exist, in case a category wasn't in the sample data
final_cols = [col for col in all_columns if col in customer_df.columns]
print(customer_df[final_cols].to_markdown(floatfmt=".2f"))

print("\n\nFinal Data Info:")
customer_df.info()